In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
import plotly.express as px
import torch
import pandas as pd
import time
from tqdm import tqdm
from glob import glob
import numpy as np
import sqlite3
from glob import glob
import matplotlib.pyplot as plt
from tqdm import tqdm
import joblib
import platform
from pathlib import Path
from bertopic import BERTopic
import pandas as pd
import plotly.io as pio




/home/students/s328743/.conda/envs/bertopic_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

if platform.node().startswith("jupyter") or "cluster" in platform.node().lower():
    # on cluster
    extracted_dir = os.path.expanduser("~/telegram_2024/usc-tg-24-us-election/extracted")
else:
    # on laptop
    extracted_dir = os.path.join("..", "material", "extracted")

print("✅ Using path:", extracted_dir)


✅ Using path: /home/students/s328743/telegram_2024/usc-tg-24-us-election/extracted


In [3]:
chats_path = '../material/chats.db'
conn = sqlite3.connect(chats_path)
cursor=conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables=cursor.fetchall()

# ========================
# 1. Leggi chats.db (SQLite)
# ========================

print("Tables in DB:", tables)
try:
    df_chats = pd.read_sql_query("SELECT * FROM chats", conn)
    df_chats =df_chats.drop_duplicates(subset='type_and_id')
    print("number of unique chats", len(df_chats))
    print("chats.db - Tabella 'chats'")
    print(df_chats.head())
except Exception as e:
    print("Errore nel leggere la tabella:", e)

conn.close()

# ========================
# 2. Leggi discovery_edges.csv.gz
# ========================
try:
    df_edges = pd.read_csv('../material/discovery_edges.csv.gz')
    df_edges = df_edges.drop_duplicates(subset='type_and_id')
    print("✅ discovery_edges.csv.gz, \n" \
    "Il timestamp da l'ultima volta che hanno visitato quel gruppo ma questo significa che non è davvero indicativo di una timeline \n")
    print(df_edges.head())
except Exception as e:
    print("Errore nel leggere discovery_edges:", e)

# ========================
# 3. Leggi first_nodes.csv.gz
# ========================
try:
    df_first_nodes = pd.read_csv('../material/first_nodes.csv.gz')
    print("number of non unique first nodes", len(df_first_nodes))
    df_first_nodes = df_first_nodes.drop_duplicates(subset='type_and_id')
    print("✅ first_nodes.csv.gz")
    print(df_first_nodes.head())
    print("number of unique first nodes", len(df_first_nodes))
except Exception as e:
    print("Errore nel leggere first_nodes:", e)

print("type_and_id unique in df_first_nodes" + str(df_first_nodes.type_and_id.nunique()))
print("type_and_id in df_first_nodes" + str(len(df_first_nodes)))
print("type_and_id NaN in df_first_nodes " + str(df_first_nodes['type_and_id'].isna().sum()))

Tables in DB: [('chats',)]
number of unique chats 127141
chats.db - Tabella 'chats'
           type_and_id                   token                  parent  \
0                 None  [keyword] thedemocrats                    None   
40  channel_1889806290         thedemocratskmf  [keyword] thedemocrats   
41  channel_1413288788       thedemocratsindia  [keyword] thedemocrats   
42  channel_1709284265           thedemocratsa  [keyword] thedemocrats   
43  channel_1807294270             democratsmv  [keyword] thedemocrats   

       timestamp  
0   1.722583e+09  
40  1.728093e+09  
41  1.728093e+09  
42  1.728093e+09  
43  1.728093e+09  
✅ discovery_edges.csv.gz, 
Il timestamp da l'ultima volta che hanno visitato quel gruppo ma questo significa che non è davvero indicativo di una timeline 

          type_and_id              parent     timestamp
0  channel_1306559115  channel_1840578235  1.722586e+09
1  channel_2036850729  channel_1840578235  1.722586e+09
2  channel_1941222046  channel_18

In [4]:
df_preprocessed_non_empty_channels = pd.read_csv(
    "../material/preprocessed_messages.tsv.gz", 
    sep='\t', 
    compression='gzip'
)

df_preprocessed_non_empty_channels_only_with_short_messages = pd.read_csv(
    "../material/preprocessed_short_messages.tsv.gz", 
    sep='\t', 
    compression='gzip'
)

df_preprocessed_non_empty_channels_spam_messages = pd.read_csv(
    "../material/preprocessed_spam_messages.tsv.gz", 
    sep='\t', 
    compression='gzip'
)

df_channels_without_message = pd.read_csv(
    "../material/channels_without_message.tsv.gz", 
    sep='\t', 
    compression='gzip'
)

df_english_preprocessed_non_empty_channels = pd.read_csv(
    "../material/preprocessed_english_messages.tsv.gz", 
    sep='\t', 
    compression='gzip'
)

df_sampled = pd.read_csv(
    "../final/df_sampled.csv"
)

print("--- len of df_sampled", len(df_sampled))
print("--- first rows of df_channels_without_message", df_channels_without_message.head(1))
print("--- first rows of df_preprocessed_non_empty_channels", df_preprocessed_non_empty_channels.head(1))
print("--- first rows of df_english_preprocessed_non_empty_channels:", df_english_preprocessed_non_empty_channels.head(1))
print("\n---\n--- number of initial seeds:", str(len(df_first_nodes)))
print("--- Unique channels in df_channels_without_message:", df_channels_without_message['channel_id'].nunique())
print("--- Number of distinct channel_ids in file_args: 180")
print("--- Distinct channels: Unique channel ids with messages in df_preprocessed_non_empty_channels(also before modification)", df_preprocessed_non_empty_channels['channel_id'].nunique())
print("--- Distinct english channels: Unique channels in df_english_preprocessed_non_empty_channels:", df_english_preprocessed_non_empty_channels['channel_id'].nunique())


--- len of df_sampled 154346
--- first rows of df_channels_without_message            channel_id
0  channel_1413288788
--- first rows of df_preprocessed_non_empty_channels                                                 text   timestamp  \
0  Welcome to the Republican Party $GOP where we ...  1719544548   

                                   text_preprocessed language  \
0  welcome republican party proudly loudly suppor...       en   

           channel_id               token                 date  
0  channel_2202860593  republicanpartyeth  2024-06-28 03:15:48  
--- first rows of df_english_preprocessed_non_empty_channels:                                                 text   timestamp  \
0  Welcome to the Republican Party $GOP where we ...  1719544548   

                                   text_preprocessed language  \
0  welcome republican party proudly loudly suppor...       en   

           channel_id               token                 date  
0  channel_2202860593  republicanpa

In [5]:
# Example structure of "../final/next/next_grid_search_results.csv"
#
# Each row = one experiment (one embedding model + one UMAP config + one HDBSCAN config).
#
# Columns:
#   model                    → which SentenceTransformer model was used
#   umap_n_components        → dimensionality of UMAP output (e.g. 3 or 5)
#   umap_n_neighbors         → neighborhood size used by UMAP
#   umap_min_dist            → minimum distance between UMAP points
#   hdbscan_min_cluster_size → minimum cluster size for HDBSCAN
#   coherence                → topic coherence score (higher = topics make more sense)
#   diversity                → topic diversity score (higher = less overlap between topics)
#   silhouette               → silhouette score of the clustering (higher = better separation)
#   n_outliers               → number of documents marked as outliers (topic = -1)
#   n_topics                 → number of discovered topics (excluding outliers)
#   min_topic                → size of the smallest topic (in number of documents)
#   max_topic                → size of the largest topic (in number of documents)
#
# Example rows (fake numbers for illustration):
#
# model,umap_n_components,umap_n_neighbors,umap_min_dist,hdbscan_min_cluster_size,coherence,diversity,silhouette,n_outliers,n_topics,min_topic,max_topic
# all-distilroberta-v1,5,25,0.0,100,0.423,0.871,0.312,150,12,18,230
# all-distilroberta-v1,3,5,0.1,50,0.401,0.856,0.295,180,10,20,210
# paraphrase-MiniLM-L6-v2,5,125,0.0,500,0.388,0.902,0.276,220,8,15,180
# all-MiniLM-L6-v2,3,25,0.1,250,0.412,0.889,0.301,140,11,19,240



df_grid = pd.read_csv("../final/grid_search_results.csv")
df_grid['avg_score'] = (df_grid['silhouette'] + df_grid['coherence'] + df_grid['diversity']) / 3
df_grid=df_grid[df_grid['hdbscan_min_cluster_size']!=500]
best_models = {
    'silhouette': df_grid.sort_values(by='silhouette', ascending=False).iloc[0],
    'coherence': df_grid.sort_values(by='coherence', ascending=False).iloc[0],
    'diversity': df_grid.sort_values(by='diversity', ascending=False).iloc[0],
    'avg_score': df_grid.sort_values(by='avg_score', ascending=False).iloc[0]
}

topic_models = {}
for key, row in best_models.items():
    suffix = f"{row['model']}_umap{row['umap_n_components']}_umap{row['umap_n_neighbors']}_umap{row['umap_min_dist']}_hdbscan{row['hdbscan_min_cluster_size']}"
    model_path = Path(f"../final/bertopic_models/{suffix}")
    
    if not model_path.exists():
        print(f"Path not found: {model_path}")
        continue
    
    print(f"Loading model for best {key} from: {model_path}")
    topic_model = BERTopic.load(model_path)
    topic_models[key] = topic_model

print("Total number of topics found by the model:", len(topic_model.topics_))


Loading model for best silhouette from: ../final/bertopic_models/all-MiniLM-L6-v2_umap5_umap5_umap0.0_hdbscan30
Loading model for best coherence from: ../final/bertopic_models/paraphrase-MiniLM-L6-v2_umap5_umap25_umap0.1_hdbscan250
Loading model for best diversity from: ../final/bertopic_models/all-distilroberta-v1_umap5_umap125_umap0.1_hdbscan250
Loading model for best avg_score from: ../final/bertopic_models/all-distilroberta-v1_umap5_umap25_umap0.0_hdbscan250
Total number of topics found by the model: 154346


In [6]:
# Show available options
print("Choose a metric to select the best model:")
print("Options: avg_score, coherence, diversity, silhouette")

choice = "coherence"

# Select the chosen model
best_model = topic_models[choice]

# Create directory for saving the model
final_model_dir = "../final/best"
os.makedirs(final_model_dir, exist_ok=True)

# Save the BERTopic model
best_model_path = os.path.join(final_model_dir, "best_model.pkl")
best_model.save(best_model_path)

# Save the associated vectorizer
best_vectorizer_path = os.path.join(final_model_dir, "best_vectorizer.pkl")
joblib.dump(best_model.vectorizer_model, best_vectorizer_path)


2025-09-23 11:38:23,081 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Choose a metric to select the best model:
Options: avg_score, coherence, diversity, silhouette


['../final/best/best_vectorizer.pkl']

- reduce outliers

In [7]:
import numpy as np

row = best_models[choice]
best_model_name = row['model']

embeddings_path = f"../final/next/next_final_embeddings_{best_model_name}.npy"
embeddings = np.load(embeddings_path)

print("Embeddings shape:", embeddings.shape)

umap_model=best_model.umap_model
reduced_embeddings=umap_model.transform(embeddings)


#Quando BERTopic prova a riassegnare un outlier, confronta il suo contenuto testuale con i topic usando una misura di similarità tra vettori TF-IDF.
#Di solito si tratta di una similarità coseno, che varia tra:
#0.0 = nessuna somiglianza
#1.0 = perfetta somiglianza
#0.1 è molto poco
new_topics = best_model.reduce_outliers(list(df_sampled['text_preprocessed']), best_model.topics_ , strategy="c-tf-idf", threshold=0.1)

# update_topics recalculates the topic representations after reassignments.
# It takes the new topic assignments (including reinserted outliers),
# re-vectorizes the documents using the provided vectorizer_model,
# and rebuilds the c-TF-IDF representation for each topic.
# The 'top_n_words' parameter controls how many of the most representative
# words per topic are stored and updated in the model.
best_model.update_topics(list(df_sampled['text_preprocessed']), topics=new_topics,
                          vectorizer_model=best_model.vectorizer_model,top_n_words=20)


Embeddings shape: (78780, 384)


2025-09-23 11:40:19,049 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


,Topic,Count,Name,Representation,Representative_Docs
0,-1,553,-1_reputation_admins_reported_vance,"[reputation, admins, reported, vance, welcome,...",[kamala harris harris ctxn holder market chart...
1,0,142930,0_trump_like_people_join,"[trump, like, people, join, read, biden, presi...",[ideology president trump biden democrat nomin...
2,1,3360,1_harris_holder_events_chart,"[harris, holder, events, chart, market, kamala...",[kamala harris harris airvbasp holder market c...
3,2,1959,2_harris_position_trending_events,"[harris, position, trending, events, chart, ma...",[kamala harris harris dflfu position market ch...
4,3,1340,3_trumpbuy_trumpprice_holder_events,"[trumpbuy, trumpprice, holder, events, chart, ...",[trumpbuy trumpfmynd holder trumpprice market ...
5,4,1034,4_trending_harris_holder_events,"[trending, harris, holder, events, chart, mark...",[kamala harris harris qsqset holder market cha...
6,5,723,5_comp_prize_ends_biggest,"[comp, prize, ends, biggest, trade, hours, cha...",[biden bidenxb position market biggest comp pr...
7,6,712,6_blocklist_automated_bites_match,"[blocklist, automated, bites, match, dust, ban...",[bites dust banned little reason automated blo...
8,7,593,7_trumpbuy_trumpprice_position_events,"[trumpbuy, trumpprice, position, events, chart...",[trumpbuy trumphdmfa position trumpprice marke...
9,8,403,8_spybox_trumponsol_alert_group,"[spybox, trumponsol, alert, group, follow, fol...",[follow alert followed trumponsol place group ...
